In [1]:
import os
import pandas as pd
import numpy as np

# Path definitions
RAW_DATA_PATH = os.path.join("..", "data", "raw", "credit_risk_dataset.csv")
PROCESSED_DATA_PATH = os.path.join("..", "data", "processed", "clean_credit_risk.csv")

def clean_loan_data(raw_path: str, output_path: str) -> pd.DataFrame:
    """
    Cleans raw credit risk dataset by handling anomalous outliers, 
    imputing missing values, and dropping duplicate records.
    """
    # Fallback paths if run from project root
    if not os.path.exists(raw_path):
        raw_path = os.path.join("data", "raw", "credit_risk_dataset.csv")
        output_path = os.path.join("data", "processed", "clean_credit_risk.csv")

    print("--- Starting Data Cleaning Pipeline ---")
    df = pd.read_csv(raw_path)
    initial_rows = len(df)
    print(f"Initial record count: {initial_rows}")

    # 1. Remove Exact Duplicate Rows
    df = df.drop_duplicates()
    duplicates_removed = initial_rows - len(df)
    print(f"Removed {duplicates_removed} duplicate rows.")

    # 2. Filter Impossible Anomalies / Outliers
    # - Age > 100 is physically impossible or bad data entry
    # - Employment length > 60 years or greater than age is impossible
    age_filter = df['person_age'] <= 100
    emp_filter = (df['person_emp_length'] <= 60) | (df['person_emp_length'].isnull())
    
    clean_df = df[age_filter & emp_filter].copy()
    anomalies_removed = len(df) - len(clean_df)
    print(f"Removed {anomalies_removed} anomalous records (unrealistic age/employment length).")

    # 3. Handle Missing Values
    # - person_emp_length: Impute missing values with Median (robust to right-skew)
    # - loan_int_rate: Impute missing values with Group Median based on loan_grade
    emp_median = clean_df['person_emp_length'].median()
    clean_df['person_emp_length'] = clean_df['person_emp_length'].fillna(emp_median)
    print(f"Imputed missing 'person_emp_length' with median: {emp_median} years")

    # Loan interest rates vary significantly by loan grade (A to G), so group imputation is more realistic
    grade_interest_medians = clean_df.groupby('loan_grade')['loan_int_rate'].transform('median')
    clean_df['loan_int_rate'] = clean_df['loan_int_rate'].fillna(grade_interest_medians)
    print("Imputed missing 'loan_int_rate' using group medians based on 'loan_grade'.")

    # 4. Final Validation
    remaining_nulls = clean_df.isnull().sum().sum()
    print(f"\nTotal null values remaining: {remaining_nulls}")
    print(f"Final clean dataset shape: {clean_df.shape[0]} rows, {clean_df.shape[1]} columns")
    print(f"Total rows retained: {(len(clean_df)/initial_rows)*100:.2f}%")

    # Save processed data
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    clean_df.to_csv(output_path, index=False)
    print(f"Saved cleaned dataset to: {output_path}")

    return clean_df

if __name__ == "__main__":
    df_clean = clean_loan_data(RAW_DATA_PATH, PROCESSED_DATA_PATH)

--- Starting Data Cleaning Pipeline ---
Initial record count: 32581
Removed 165 duplicate rows.
Removed 7 anomalous records (unrealistic age/employment length).
Imputed missing 'person_emp_length' with median: 4.0 years
Imputed missing 'loan_int_rate' using group medians based on 'loan_grade'.

Total null values remaining: 0
Final clean dataset shape: 32409 rows, 12 columns
Total rows retained: 99.47%
Saved cleaned dataset to: ..\data\processed\clean_credit_risk.csv
